# <div style="color:blue;display:inline-block;border-radius:5px;background-color:#F0E68C;font-family:Nexa;overflow:hidden"><p style="padding:15px;color:blue;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b> </b> Import Modules</p></div>

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
import os

from wordcloud import WordCloud
import warnings
warnings.filterwarnings("ignore")

# <div style="color:blue;display:inline-block;border-radius:5px;background-color:#F0E68C;font-family:Nexa;overflow:hidden"><p style="padding:15px;color:blue;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b> Load the Dataset</p></div>


In [ ]:

# Paths
csv_files = [
    "csv/Alibaba.csv",
    "csv/Aliexpress.csv",
    "csv/Myntra.csv",
    "csv/Flipkart.csv",
    "csv/Meesho.csv",
    "csv/Lazada.csv",
    "csv/Amazon shopping.csv",
    "csv/Snapdeal.csv",
    "csv/Shein.csv",
    "csv/Daraz Online Shopping App.csv",
    "csv/Walmart.csv"
]
json_files = [
    "json/Alibaba.json",
    "json/Aliexpress.json",
    "json/Myntra.json",
    "json/Flipkart.json",
    "json/Meesho.json",
    "json/Lazada.json",
    "json/Amazon shopping.json",
    "json/Snapdeal.json",
    "json/Shein.json",
    "json/Daraz Online Shopping Appjson.json",
    "json/Walmart.json"
]

# Load CSV
csv_data = {os.path.basename(f).split('.')[0]: pd.read_csv(f) for f in csv_files}

# Load JSON
json_data = {}
for f in json_files:
    with open(f, 'r') as file:
        json_data[os.path.basename(f).split('.')[0]] = pd.json_normalize(json.load(file))


In [ ]:
csv_data["Alibaba"].head()

In [ ]:
# Load CSV files into DataFrame
csv_data = [pd.read_csv(file) for file in csv_files]

# Load JSON files into DataFrame
json_data = [pd.DataFrame(json.load(open(file))) for file in json_files]

# Combine CSV and JSON data
combined_df = pd.concat(csv_data + json_data, ignore_index=True)

In [ ]:
import missingno as msno

fig, ax = plt.subplots(2,2,figsize=(12,7))
axs = np.ravel(ax)
msno.matrix(combined_df,  fontsize=9, color=(0.25,0,0.5),ax=axs[0]);
msno.bar(combined_df, fontsize=8, color=(0.25,0,0.5), ax=axs[1]);
msno.heatmap(combined_df,fontsize=8,ax=axs[2]);
msno.dendrogram(combined_df,fontsize=8,ax=axs[3], orientation='top')

fig.suptitle('Missing Values Analysis', y=1.01, fontsize=15)

# Save the plot
plt.savefig('missing_values_analysis.png')

# Show the plot
plt.show()

# <div style="color:blue;display:inline-block;border-radius:5px;background-color:#F0E68C;font-family:Nexa;overflow:hidden"><p style="padding:15px;color:blue;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b> Data Cleaning</p></div>

In [ ]:
print(combined_df.columns)

In [ ]:
combined_df.info()

In [ ]:
# Check for missing values
print(combined_df.isnull().sum())

In [ ]:
# Drop rows where reviews or ratings are missing
combined_df = combined_df.dropna(subset=['content', 'score'])

# Verify the result
print(combined_df.isnull().sum())


In [ ]:
# Ensure 'score' is treated as a categorical variable
combined_df['score'] = combined_df['score'].astype(int)

# Preview the data
combined_df[['appName', 'score']].head()


# <div style="color:blue;display:inline-block;border-radius:5px;background-color:#F0E68C;font-family:Nexa;overflow:hidden"><p style="padding:15px;color:blue;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b> Exploratory Data Analysis (EDA)📊</p></div>

In [ ]:
# Get a list of unique apps
apps = combined_df['appName'].unique()

# Define the subplot grid (e.g., 3 rows x 4 columns)
rows, cols = 3, 4
fig, axes = plt.subplots(rows, cols, figsize=(20, 15), sharey=True)
axes = axes.flatten()

# Plot countplots for each app
for i, app in enumerate(apps):
    ax = axes[i]
    app_data = combined_df[combined_df['appName'] == app]
    
    sns.countplot(
        data=app_data, 
        x='score', 
        ax=ax, 
        palette='viridis', 
        order=sorted(app_data['score'].unique())
    )
    ax.set_title(app, fontsize=12)
    ax.set_xlabel('Rating', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    
    # Annotate bar heights
    for p in ax.patches:
        height = p.get_height()
        if height > 0:  # Only annotate bars with a height greater than 0
            ax.text(
                p.get_x() + p.get_width() / 2.,
                height + 1,
                f'{int(height)}',
                ha='center',
                fontsize=9
            )

# Remove unused subplots (if any apps < rows*cols)
for j in range(len(apps), len(axes)):
    fig.delaxes(axes[j])

# Add a global title
plt.suptitle('Grouped Countplot Subplots: Ratings Distribution Across Apps', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


# <font size="+3" color='#059c99'><b> Analyze the Best-Performing Apps</b></font>

In [ ]:
# Define sentiment labels
def categorize_sentiment(score):
    if score >= 4:
        return 'Positive'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Negative'

# Apply sentiment categorization
combined_df['sentiment'] = combined_df['score'].apply(categorize_sentiment)

# Count sentiments per app
sentiment_counts = (
    combined_df.groupby(['appName', 'sentiment'])
    .size()
    .reset_index(name='count')  # Reset index to get 'count' as a column
)

# Calculate proportions per app
sentiment_counts['proportion'] = sentiment_counts.groupby('appName')['count'].transform(lambda x: x / x.sum())

# Plotting
plt.figure(figsize=(16, 10))
sns.barplot(data=sentiment_counts, x='appName', y='proportion', hue='sentiment', palette='coolwarm')
plt.title('Sentiment Distribution Across Apps', fontsize=16)
plt.xlabel('App Name', fontsize=12)
plt.ylabel('Proportion of Reviews', fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.legend(title='Sentiment', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
# Calculate the percentage of positive reviews per app
app_performance = combined_df.groupby('appName')['sentiment'].apply(lambda x: (x == 'positive').mean() * 100).reset_index(name='positive_percentage')

# Identify top 3 apps with the highest percentage of positive reviews
top_apps = app_performance.sort_values('positive_percentage', ascending=False).head(3)['appName'].values
print("Top Performing Apps:", top_apps)

# Extract reviews for these top apps
top_apps_reviews = combined_df[combined_df['appName'].isin(top_apps)]

# Group by app name and sentiment to analyze trends
top_app_trends = top_apps_reviews.groupby(['appName', 'sentiment']).size().unstack(fill_value=0)
print(top_app_trends)


# <div style="color:blue;display:inline-block;border-radius:5px;background-color:#F0E68C;font-family:Nexa;overflow:hidden"><p style="padding:15px;color:blue;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b> Dynamic Sentiment Analysis </p></div>

- We will use a popular NLP package, VADER, to dynamically classify the sentiment of reviews based on their content. VADER work well with social media-type text, which includes informal language and slang.

# <font size="+2" color='#059c99'><b> Install the required libraries</b></font>

- We need `nltk` and `vaderSentiment`

In [ ]:
pip install nltk vaderSentiment

# <font size="+3" color='#059c99'><b> Sentiment Analysis with VADER</b></font>

- We'll create a function to classify the sentiment of each review using VADER and apply it to the `content` column.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize VADER Sentiment Analyzer
analyzer = SentimentIntensityAnalyzer()

# Function to classify sentiment based on VADER scores
def classify_sentiment(review):
    sentiment_score = analyzer.polarity_scores(review)
    compound_score = sentiment_score['compound']
    
    # Classify sentiment based on compound score
    if compound_score >= 0.05:
        return 'Positive'
    elif compound_score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

# Apply sentiment classification to the review content
combined_df['sentiment'] = combined_df['content'].apply(classify_sentiment)


# <font size="+2" color='#059c99'><b> Review the output </b></font>

In [ ]:
combined_df[['content', 'sentiment']].head()

# <font size="+2" color='#059c99'><b> Clustering Apps Based on Review Characteristics </b></font>

- Now, let's use K-Means clustering to group apps based on review length and sentiment. For sentiment, we'll convert the sentiment labels into numerical values (`Positive=1`,`Neutral =0`, `Negative = -1`).

In [ ]:
sentiment_map = {'Positive': 1, 'Neutral': 0, 'Negative': -1}
combined_df['sentiment_num'] = combined_df['sentiment'].map(sentiment_map)

# <font size="+2" color='#059c99'><b> Select features for clustering</b></font>

- We will use the review length and sentiment as features for clustering

In [ ]:
# Create 'review_length' column based on the length of each review
combined_df['review_length'] = combined_df['content'].apply(len)

if 'sentiment_num' not in combined_df.columns:
    sentiment_mapping = {'positive': 1, 'negative': -1, 'neutral': 0}
    combined_df['sentiment_num'] = combined_df['sentiment'].map(sentiment_mapping)

# Now you can extract 'review_length' and 'sentiment_num' for analysis
X = combined_df[['review_length', 'sentiment_num']]

# Preview the result
print(X.head())


# <font size="+2" color='#059c99'><b> Normalize the features </b></font>

- Clustering often performs better when the data is normalized.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# <font size="+2" color='#059c99'><b> K-Means clustering </b></font>

In [ ]:
from sklearn.cluster import KMeans

# Choose the number of clusters (e.g., 3 clusters)
kmeans = KMeans(n_clusters=3, random_state=42)
combined_df['cluster'] = kmeans.fit_predict(X_scaled)


# <font size="+2" color='#059c99'><b> Review cluster results</b></font>

- Now, you can inspect how the apps are clustered based on review length and sentiment.

In [ ]:
combined_df[['appName', 'review_length', 'sentiment', 'cluster']].head()


In [ ]:
# Aggregate sentiment counts per app
sentiment_counts = combined_df.groupby(['appName', 'sentiment']).size().reset_index(name='count')

# Plotting
plt.figure(figsize=(16, 8))
sns.barplot(data=sentiment_counts, x='appName', y='count', hue='sentiment', palette='coolwarm')

# Customizing the plot
plt.title('Sentiment Distribution Across Apps', fontsize=16)
plt.xlabel('App Name', fontsize=12)
plt.ylabel('Review Count', fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.legend(title='Sentiment', loc='upper right')

# Display the plot
plt.tight_layout()
plt.show()
